# Hedging an insurance portfolio

Compare no hedge, a mortality-adjusted static hedge and a dynamic delta hedge on common realized scenarios. Every model and numerical input is written below; initial premium and deltas are computed by the chosen valuation method.

These examples use paper-calibrated financial/carbon inputs and an illustrative five-year contract. The small numerical budgets demonstrate the workflow; they are not a convergence study or the paper's full hedging experiment.

**Workspace and execution controls.** Save this notebook in the repository root. Start it from that folder or a subdirectory and select the `.venv-analysis` Python kernel described in [INSTALLATION.md](INSTALLATION.md). The setup detects the project root and locates the compiled C++ runner. `RUN_SIMULATIONS=False` reloads a completed run with matching simulation inputs. Use a new `RUN_NAME` when changing inputs; `OVERWRITE_EXISTING=True` explicitly permits replacing that run's generated files. All paths and switches below are editable.

In [ ]:
from pathlib import Path
from copy import deepcopy
import json
import sys

SEARCH_START = Path.cwd().resolve()
PROJECT_ROOT = next(
    (parent for parent in (SEARCH_START, *SEARCH_START.parents)
     if (parent / 'include' / 'carbonfund').is_dir()
     and (parent / 'python' / 'carbonfund_experiments.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start the notebook from the CarbonFund_UL folder or a subdirectory.')
BINARY = PROJECT_ROOT / 'build-paper-release' / ('carbonfund_experiment.exe' if sys.platform == 'win32' else 'carbonfund_experiment')
OUTPUT_ROOT = PROJECT_ROOT / 'results' / 'hedging_example'
RUN_NAME = 'cir_T5_01'
RUN_SIMULATIONS = True
OVERWRITE_EXISTING = False
SAVE_FIGURES = True
SHOW_FIGURES = True
PRINT_PROGRESS = True

sys.path.insert(0, str(PROJECT_ROOT / 'python'))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import carbonfund_experiments as cfe

RUN_DIRECTORY = OUTPUT_ROOT / RUN_NAME
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in ('.', '..'):
    raise ValueError('RUN_NAME must be one directory name.')
if RUN_SIMULATIONS and not BINARY.is_file():
    raise FileNotFoundError(f'Build carbonfund_experiment first: {BINARY}')
print('Python:', sys.executable)
print('Python interface:', Path(cfe.__file__).resolve())
print('C++ backend:', BINARY)
print('Output directory:', RUN_DIRECTORY)

**Market inputs.** The dimension is fixed at four risky assets. `risk_free_rate` and the four `expected_returns` are continuously compounded annual rates; `volatilities` are annual diffusion coefficients. The symmetric positive-definite `correlations` matrix has unit diagonal. The backend constructs the covariance from volatilities and correlations. Asset expected returns enter the fund allocation; valuation uses the risk-free financial drift.

In [ ]:
market = {
    'risk_free_rate': 0.05,
    'expected_returns': [0.25, 0.15, 0.10, 0.08],
    'volatilities': [0.30, 0.25, 0.20, 0.16],
    'correlations': [
        [1.0000, 0.4397, 0.3900, 0.3168],
        [0.4397, 1.0000, 0.2954, 0.3261],
        [0.3900, 0.2954, 1.0000, 0.3134],
        [0.3168, 0.3261, 0.3134, 1.0000],
    ],
}

**Carbon inputs and model choices.** `model` selects `CIR`, `OU` or `GeoOU`. All vectors have four entries, one per factor. CIR solves $dC_i=\kappa_i(\bar C_i-C_i)dt+\sigma_i\sqrt{C_i}\,dB_i$; `initial` is the initial intensity, `long_run_mean` is $\bar C$, `mean_reversion` is $\kappa$, and `volatility` is $\sigma$. This scheme requires nonnegative states and $\kappa_i\bar C_i\geq\sigma_i^2/4$.

OU uses the same input names with additive Gaussian noise and allows negative intensities; allocation uses their positive part. GeoOU applies OU dynamics to log intensity: change `model` to `GeoOU`, replace `long_run_mean` with `long_run_log_mean`, and supply the desired log means. GeoOU `initial` remains in positive level units. Volatility units differ between models, so model changes require corresponding parameter choices. Carbon factors and financial noise are independent.

In [ ]:
carbon = {
    'model': 'CIR',
    'initial': [5000.0, 4000.0, 3000.0, 1000.0],
    'long_run_mean': [2500.0, 2000.0, 1500.0, 500.0],
    'mean_reversion': [0.05, 0.05, 0.05, 0.05],
    'volatility': [3.0, 3.0, 3.0, 3.0],
}

**Fund preferences.** `risk_aversion` is the positive quadratic-risk parameter $\delta$, and `carbon_penalty` is the nonnegative carbon-intensity penalty $\alpha$. They determine the fund's investment allocation together with market inputs and current carbon intensities. The remainder of wealth is invested in the risk-free account.

In [ ]:
fund = {
    'risk_aversion': 1.0,
    'carbon_penalty': 0.0025,
}

**Contract inputs.** `fund_value` is the initial fund value $X_0$ and `maturity` is in years. The lower and upper guarantees start at `minimum_guarantee` and `maximum_guarantee` and grow at the two specified continuous rates. The benefit at payment time is the fund value bounded by those guarantees. The bounds below coincide at time zero and separate for positive times.

PE pays on survival to maturity; TI pays on death before maturity; EI combines the PE survival benefit and the TI death benefit multiplied by `endowment_death_benefit_fraction`. Benefits are discounted once. Guarantees remain fixed when differentiating with respect to the current fund value.

In [ ]:
contract = {
    'fund_value': 1.0,
    'maturity': 5.0,
    'minimum_guarantee': 1.0,
    'maximum_guarantee': 1.0,
    'minimum_guarantee_growth_rate': 0.05,
    'maximum_guarantee_growth_rate': 0.50,
    'endowment_death_benefit_fraction': 1.0,
}

**Mortality inputs.** For initial age $a$, the Gompertz-Makeham hazard is

$$\mu(a+t)=\xi+\frac{1}{b}\exp\!\left(\frac{a+t-m}{b}\right).$$

`xi` is the nonnegative Makeham constant, `m` the Gompertz location and `b > 0` its scale in years. The corrected paper calibration uses `xi = 0.0041959`. Mortality is independent of financial and carbon risk.

In [ ]:
mortality = {
    'xi': 0.0041959,
    'm': 79.6921211,
    'b': 11.5818911,
}

**Population inputs.** Every group below chooses a contract type, initial age, number of individual lives and total group notional. The notebook expands each group into distinct equal-notional policies. Groups may differ in type, age and notional. All policies share the contract terms, and their lifetimes are independent. Total initial exposure is the sum of the group notionals; initial premium and deltas are computed for this actual population.

In [ ]:
population_groups = [
    {'type': 'EI', 'age': 60.0, 'count': 100, 'total_notional': 1.0},
]
population = []
for group in population_groups:
    count = group['count']
    if isinstance(count, bool) or not isinstance(count, int) or count < 1:
        raise ValueError('Each group count must be a positive integer.')
    population.extend(
        {'type': group['type'], 'age': group['age'],
         'notional': group['total_notional'] / count}
        for _ in range(count)
    )
print('Policies:', len(population))
print('Total notional:', sum(p['notional'] for p in population))

**Outer simulation inputs.** Each outer scenario contains a common carbon/fund path and realized deaths for the population. `outer_scenarios` is the sample size of the cost distribution. `intervals` controls market discretization, hedge rebalancing and death-settlement dates. These are separate from inner pricing intervals.

`threads` requests parallel outer workers in C++; `0` uses the OpenMP runtime limit and `1` selects one worker. `progress_every=10` prints after each ten completed scenarios and at completion; `0` disables these messages. It does not change results or save partial scenario checkpoints. The carbon, financial and mortality streams have separate seeds.

In [ ]:
outer_simulation = {
    'outer_scenarios': 40,
    'intervals': 100,
    'progress_every': 10,
    'threads': 0,
    'carbon_seed': 2026101801,
    'financial_seed': 2026101802,
    'mortality_seed': 2026101803,
}

**Initial valuation inputs.** Compute the portfolio premium and aggregate PE/TI/EI deltas once at time zero. `paths` and `intervals` control this independent valuation budget. `configuration` selects one of the four pricing estimators. The initial carbon and financial seeds are distinct from all other streams, including when the financial stream is unused by a conditioned estimator.

In [ ]:
initial_valuation = {
    'paths': 512,
    'intervals': 100,
    'configuration': 'conditioned_antithetic',
    'carbon_seed': 2026101804,
    'financial_seed': 2026101805,
}

**Revaluation inputs.** At each intermediate outer date, reprice the surviving portfolio and compute its new aggregate delta from the current state. The remaining inner grid uses

$$N(t)=\max\!\left(N_{\min},\left\lceil N_{\mathrm{base}}(T-t)/T\right\rceil\right).$$

`intervals` is $N_{\mathrm{base}}$ and `minimum_intervals` is $N_{\min}$, between 1 and `intervals`. Setting both equal fixes the interval count. `paths`, `configuration` and the two seeds control subsequent valuations. All seven seed inputs across outer, initial and revaluation sections must be distinct. Changing this budget does not change realized market or death scenarios.

In [ ]:
revaluation = {
    'paths': 32,
    'intervals': 40,
    'minimum_intervals': 4,
    'configuration': 'conditioned_antithetic',
    'carbon_seed': 2026101806,
    'financial_seed': 2026101807,
}

**Analysis inputs.** `normalization` selects aggregate, per-policy or per-notional units; raw simulation files retain aggregate amounts. Means and standard errors divide by exposure; variances divide by exposure squared. `confidence_level` selects approximate Student t mean intervals. `comparison_family_size=0` uses the current run's contrast count; a larger declared family gives Bonferroni simultaneous intervals.

Quantiles and upper-tail expected-shortfall levels are probabilities strictly between zero and one. `bins`, `plot_range` and `zoom` control histograms and the optional inset. Figure formats, raster DPI and title are explicit. These settings can be changed when reloading saved results without rerunning C++.

In [ ]:
analysis = {
    'normalization': 'per_notional',
    'confidence_level': 0.95,
    'comparison_family_size': 0,
    'quantiles': [0.025, 0.50, 0.90, 0.95, 0.975, 0.99],
    'expected_shortfall_levels': [0.95, 0.99],
    'bins': 70,
    'zoom': None,
    'plot_range': None,
    'figure_formats': ['png', 'pdf'],
    'dpi': 180,
    'title': '',
}

**Configuration.** Assemble one complete schema-version-1 dictionary from the explicit input cells. Deep copies keep the run inputs separate from later edits. Validation catches inconsistent dimensions, model inputs, estimator names, stream seeds and analysis settings before launching the backend. The exact configuration is saved with the run.

In [ ]:
NOTES = 'Explicit hedging notebook example; illustrative numerical budgets.'
config = {
    'schema_version': 1,
    'experiment': 'hedging',
    'market': deepcopy(market),
    'carbon': deepcopy(carbon),
    'fund': deepcopy(fund),
    'contract': deepcopy(contract),
    'mortality': deepcopy(mortality),
    'population': deepcopy(population),
    'hedging': {
        **deepcopy(outer_simulation),
        'initial': deepcopy(initial_valuation),
        'revaluation': deepcopy(revaluation),
    },
    'analysis': deepcopy(analysis),
    'notes': NOTES,
}
cfe.validate_config(config)
print(json.dumps(config, indent=2))

**Run or reload.** Simulation is performed by the selected C++ executable. The interface writes the configuration, metadata, raw numerical output, statistical summaries and execution log. Existing nonempty output directories require explicit overwrite permission.

Reload mode checks that simulation inputs match the saved run; only analysis settings and descriptive notes may differ. Figures are generated in the final cell so their saving and display are controlled explicitly.

In [ ]:
if RUN_SIMULATIONS:
    result = cfe.run_experiment(
        config, RUN_DIRECTORY, binary=BINARY,
        overwrite=OVERWRITE_EXISTING, plots=False, progress=PRINT_PROGRESS,
    )
else:
    result = cfe.load_result(RUN_DIRECTORY)
    def simulation_inputs(cfg):
        return {key: value for key, value in cfg.items()
                if key not in ('analysis', 'notes')}
    if simulation_inputs(result.config) != simulation_inputs(config):
        raise ValueError('Current simulation inputs differ from the saved run.')
    result.config['analysis'] = deepcopy(analysis)
    result.save_analysis(plots=False)
print('Saved results:', result.directory)

**Hedging results.** All three costs are present values at time zero, with positive values indicating additional funding. The no-hedge strategy holds the premium in cash. The static benchmark adjusts its initial aggregate deltas for surviving-notional ratios by contract type. The dynamic strategy recomputes delta at hedge dates. All three share the initial premium and realized market/death scenario.

The table reports means, sample variances, standard deviations, mean standard errors and intervals, tails and positive-cost probabilities. Paired strategy differences use common scenario rows. Mean intervals condition on the initial premium used and omit its separate estimation uncertainty and discretization error. Inspect actual workers and elapsed time below when assessing performance.

In [ ]:
display(result.summary())
display(result.comparisons())
run = result.metadata['run']
print('Initial premium:', run['initial_price'])
print('Initial deltas PE/TI/EI:', run['initial_deltas_PE_TI_EI'])
print('Requested/actual threads:', run['threads_requested'], run['threads_used'])
print('Initial elapsed seconds:', run['initial_wall_seconds'])
print('Outer elapsed seconds:', run['simulation_wall_seconds'])
print('Raw scenario CSV:', result.directory / 'hedging.csv')
print('Summary CSV:', result.directory / 'summary.csv')

**Figures and exported files.** Create the configured plots from saved numerical results. `SAVE_FIGURES` writes the selected formats and `SHOW_FIGURES` displays them. The original CSV/JSON outputs remain available for MATLAB, R or later Python sessions. Reloading with `load_result` never invokes the C++ simulation. Plot ranges affect presentation; the summary statistics are computed from the complete retained sample.

In [ ]:
if SAVE_FIGURES or SHOW_FIGURES:
    figure = result.plot()
    if SAVE_FIGURES:
        for extension in analysis['figure_formats']:
            figure.savefig(
                result.directory / ('distributions.' + extension),
                dpi=analysis['dpi'],
            )
    if SHOW_FIGURES:
        plt.show()
    plt.close(figure)
for path in sorted(result.directory.iterdir()):
    if path.is_file():
        print(path.name)